# Recipe writing: an adapter efficiency benchmark
**Name: Priyanshi Goyal · Roll number: A24ARIU0009**

A new sequential experiment design compares original GPT-2, full fine-tuning and LoRA ranks 4 and 8.
It uses grouped dish splits, answer-only loss, identical generation prompts, parameter counts,
wall-clock time and saved evaluation examples. Models are released between experiments to avoid
keeping several GPT-2 copies on the GPU. The rank comparison changes rank alone.

Task and data provenance: [course recipe lab](https://github.com/bk-anupam/generative_ai_shared/tree/master/module3/labs/gpt2_lora_recipes).
Data: K. Jain, [Indian Food Recipes, Mendeley V1](https://data.mendeley.com/datasets/xsphgmmh7b/1),
CC BY 4.0, sourced from Archana's Kitchen. This implementation retains dataset download identity
and dish-grouping principles from the course, with a rewritten pipeline and experiment organization.
**No model scores have been measured here.**


## Setup and data contract
Install `torch transformers accelerate peft pandas scikit-learn rouge_score sacrebleu` in a fresh
Python environment. A CUDA GPU is recommended. Downloading GPT-2 and the data requires internet.
`TRAIN_LIMIT=200` is a smoke run only; set it to `None` for the complete filtered training split.
Unlike the course's cuisine blacklist, this study retains all English recipes in the source; comparisons
are internal to this notebook and should not be treated as reproducing the course baseline exactly.


In [ ]:
import os, json, time, random, gc, math
from pathlib import Path
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
import numpy as np
import pandas as pd
import torch
import transformers
from transformers import AutoTokenizer, Trainer, TrainingArguments
SEED = 42
transformers.set_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
ROOT_OUT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
print({'student': 'Priyanshi Goyal', 'roll': 'A24ARIU0009',
       'torch': torch.__version__, 'transformers': transformers.__version__, 'device': DEVICE})


In [ ]:
import re, hashlib, urllib.request
from sklearn.model_selection import GroupShuffleSplit
OUT=ROOT_OUT/'priyanshi_recipes_A24ARIU0009'
OUT.mkdir(parents=True,exist_ok=True)
CHECKPOINT='openai-community/gpt2'
TRAIN_LIMIT=200
EPOCHS=2
MAX_LENGTH=768
MAX_NEW=192
CSV=OUT/'IndianFoodDatasetCSV.csv'
URL='https://data.mendeley.com/public-files/datasets/xsphgmmh7b/files/6fb5ae32-a815-4b8f-9d68-8d512227eac8/file_downloaded'
if not CSV.exists():
    request=urllib.request.Request(URL,headers={'User-Agent':'recipe-lab'})
    with urllib.request.urlopen(request,timeout=120) as response: CSV.write_bytes(response.read())
assert hashlib.sha256(CSV.read_bytes()).hexdigest()=='273f8d45f41aca9af0396dda07af137c68c7387a1506992904144b944c4346be'
raw=pd.read_csv(CSV,encoding='utf-8-sig')
fields=['RecipeName','Ingredients','Instructions','Cuisine','Course','Diet']
assert set(fields)<=set(raw)
rows=raw.dropna(subset=fields).copy()
for column in fields: rows[column]=rows[column].astype(str).str.replace('\ufeff','',regex=False).str.strip()
rows=rows[(rows[fields].ne('')).all(axis=1)]
rows=rows[~rows[['RecipeName','Ingredients','Instructions']].apply(lambda s:s.str.contains(r'[ऀ-ॿ]')).any(axis=1)]

def dish_group(name):
    stem=re.split(r'\s[-–]\s|\(',name, maxsplit=1)[0].lower()
    stem=re.sub(r'\brecipe\b','',stem)
    return ' '.join(re.sub(r'[^a-z0-9 ]',' ',stem).split())

rows['group']=rows.RecipeName.map(dish_group)
assert dish_group('Aamras Recipe - Ripe Mango Ras')==dish_group('Aamras Recipe')
train_idx,rest_idx=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=SEED).split(rows,groups=rows.group))
train_rows,rest=rows.iloc[train_idx],rows.iloc[rest_idx]
val_idx,test_idx=next(GroupShuffleSplit(n_splits=1,test_size=.5,random_state=SEED).split(rest,groups=rest.group))
val_rows,test_rows=rest.iloc[val_idx],rest.iloc[test_idx]
assert set(train_rows.group).isdisjoint(set(val_rows.group)|set(test_rows.group))
assert set(val_rows.group).isdisjoint(test_rows.group)
pd.concat([part.assign(split=name) for name,part in [('train',train_rows),('validation',val_rows),('test',test_rows)]])[['RecipeName','group','split']].to_csv(OUT/'splits.csv',index=False)
if TRAIN_LIMIT is not None: train_rows=train_rows.sample(n=min(TRAIN_LIMIT,len(train_rows)),random_state=SEED)
tokenizer=AutoTokenizer.from_pretrained(CHECKPOINT)
tokenizer.pad_token=tokenizer.eos_token
print({'train':len(train_rows),'validation':len(val_rows),'untouched_test':len(test_rows)})


In [ ]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from rouge_score import rouge_scorer

def pack_answer(prefix, answer, stop_id):
    if not prefix or not answer: raise ValueError('Empty prompt or answer')
    target=answer+[stop_id]
    ids=prefix+target
    return {'input_ids':ids,'attention_mask':[1]*len(ids),'labels':[-100]*len(prefix)+target}

def collate_answers(batch):
    width=max(len(row['input_ids']) for row in batch)
    output={key:torch.full((len(batch),width),fill,dtype=torch.long)
            for key,fill in [('input_ids',tokenizer.pad_token_id),('attention_mask',0),('labels',-100)]}
    for i,row in enumerate(batch):
        n=len(row['input_ids'])
        for key in output: output[key][i,:n]=torch.tensor(row[key],dtype=torch.long)
    return output

@torch.inference_mode()
def decode_prompts(model, prompts, max_new):
    model.eval()
    predictions=[]
    for prompt in prompts:
        ids=torch.tensor([prompt],dtype=torch.long,device=model.device)
        result=model.generate(input_ids=ids,attention_mask=torch.ones_like(ids),
            do_sample=False,max_new_tokens=max_new,pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id)
        predictions.append(tokenizer.decode(result[0,len(prompt):],skip_special_tokens=True).strip())
    return predictions

def rouge_table(predictions, references):
    assert len(predictions)==len(references)>0
    scorer=rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'],use_stemmer=True)
    pairs=[scorer.score(reference,prediction) for prediction,reference in zip(predictions,references)]
    return {key:100*np.mean([row[key].fmeasure for row in pairs]) for key in ['rouge1','rouge2','rougeL']}

def fit_model(model, args, train_data, validation=None):
    trainer=Trainer(model=model,args=args,train_dataset=train_data,eval_dataset=validation,
                    data_collator=collate_answers)
    if DEVICE=='cuda': torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
    started=time.perf_counter()
    trainer.train()
    if DEVICE=='cuda': torch.cuda.synchronize()
    stats={'train_seconds':time.perf_counter()-started,
           'trainable_parameters':sum(p.numel() for p in model.parameters() if p.requires_grad),
           'peak_gpu_gb':torch.cuda.max_memory_allocated()/1e9 if DEVICE=='cuda' else None}
    if validation is not None: stats['validation_loss']=trainer.evaluate()['eval_loss']
    Path(args.output_dir).mkdir(parents=True,exist_ok=True)
    Path(args.output_dir,'training_log.json').write_text(json.dumps(trainer.state.log_history,indent=2))
    from accelerate.utils import extract_model_from_parallel
    extract_model_from_parallel(model, keep_fp32_wrapper=False)
    return trainer,stats

# A padding check explicitly preserves real EOS supervision even when PAD == EOS.
probe=collate_answers([pack_answer([3,4],[5],tokenizer.eos_token_id),pack_answer([3],[6],tokenizer.eos_token_id)])
assert probe['labels'][0,-1].item()==tokenizer.eos_token_id
assert probe['labels'][1,-1].item()==-100 and probe['attention_mask'][1,-1].item()==0
assert (probe['labels'][:,0]==-100).all()


In [ ]:
def recipe_prompt(row):
    return (f"Write a cooking method for {row['RecipeName']}.\n"
            f"Cuisine: {row['Cuisine']}; course: {row['Course']}; diet: {row['Diet']}.\n"
            f"Ingredients: {row['Ingredients']}\nCooking method:\n")

def prompt_tokens(row):
    return tokenizer.encode(recipe_prompt(row),add_special_tokens=False)

def recipe_example(row):
    return pack_answer(prompt_tokens(row),tokenizer.encode(row['Instructions'],add_special_tokens=False),tokenizer.eos_token_id)

def prepare(rows):
    examples=[recipe_example(row) for row in rows.to_dict('records')]
    kept=[row for row in examples if len(row['input_ids'])<=MAX_LENGTH]
    print({'kept':len(kept),'dropped_long':len(examples)-len(kept)})
    if not kept: raise ValueError('No complete examples fit the token limit')
    return kept

train_data,val_data=prepare(train_rows),prepare(val_rows)
evaluation=[r for r in val_rows.to_dict('records') if len(prompt_tokens(r))+MAX_NEW<=1024][:40]
assert evaluation
prompts=[prompt_tokens(r) for r in evaluation]
references=[r['Instructions'] for r in evaluation]
(OUT/'evaluation_ids.json').write_text(json.dumps([r['RecipeName'] for r in evaluation],indent=2))


## Sequential benchmark
For GPT-2's fused `c_attn` projection, each layer maps 768 dimensions to 2304. Rank r adds
r × (768 + 2304) parameters per layer: 36,864r across 12 layers. Rank 8 therefore trains
294,912 adapter parameters. GPT-2 uses Conv1D-style transposed weights, so `fan_in_fan_out=True`.
Both LoRA ranks share learning rate and schedule; full fine-tuning uses a smaller learning rate,
so the full-versus-LoRA comparison is a practical recipe comparison rather than a one-factor ablation.


In [ ]:
import sacrebleu
from peft import PeftModel
experiments=[('original',None,None),('full',None,5e-5),('lora_r4',4,5e-4),('lora_r8',8,5e-4)]
results=[]
GENERAL_TEXTS=[
    'The morning train arrived at the station. Passengers walked across the platform to meet their families.',
    'Water evaporates from lakes and oceans. Clouds form when the rising air cools and the vapor condenses.'
]

@torch.inference_mode()
def general_loss(model):
    model.eval()
    weighted,total=0.0,0
    for text in GENERAL_TEXTS:
        batch=tokenizer(text,return_tensors='pt').to(model.device)
        n=batch.input_ids.shape[1]-1
        weighted+=model(**batch,labels=batch.input_ids).loss.item()*n
        total+=n
    return weighted/total

for name,rank,lr in experiments:
    transformers.set_seed(SEED)
    model=AutoModelForCausalLM.from_pretrained(CHECKPOINT).to(DEVICE)
    stats={'train_seconds':0.0,'trainable_parameters':0,'peak_gpu_gb':None}
    if rank:
        model=get_peft_model(model,LoraConfig(task_type='CAUSAL_LM',r=rank,lora_alpha=2*rank,
            lora_dropout=.05,target_modules=['c_attn'],fan_in_fan_out=True,bias='none'))
        assert sum(p.numel() for p in model.parameters() if p.requires_grad)==12*rank*(768+2304)
    if lr is not None:
        args=TrainingArguments(output_dir=str(OUT/name),num_train_epochs=EPOCHS,
            per_device_train_batch_size=2,gradient_accumulation_steps=4,per_device_eval_batch_size=2,
            learning_rate=lr,eval_strategy='epoch',save_strategy='no',logging_steps=10,
            report_to='none',seed=SEED,fp16=DEVICE=='cuda',remove_unused_columns=False)
        trainer,stats=fit_model(model,args,train_data,val_data)
        del trainer
    predictions=decode_prompts(model,prompts,MAX_NEW)
    result={'model':name,**stats,**rouge_table(predictions,references),
            'BLEU':sacrebleu.corpus_bleu(predictions,[references]).score,
            'general_text_loss':general_loss(model)}
    results.append(result)
    pd.DataFrame({'dish':[r['RecipeName'] for r in evaluation],'reference':references,'prediction':predictions}).to_csv(OUT/f'{name}_examples.csv',index=False)
    if rank:
        model.save_pretrained(OUT/name/'adapter')
        tokenizer.save_pretrained(OUT/name/'adapter')
        model.eval()
        probe_ids=torch.tensor([prompts[0]],device=model.device)
        with torch.inference_mode(): before=model(input_ids=probe_ids).logits.float().cpu()
        del model
        gc.collect()
        if DEVICE=='cuda': torch.cuda.empty_cache()
        model=PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained(CHECKPOINT),OUT/name/'adapter').to(DEVICE).eval()
        with torch.inference_mode(): after=model(input_ids=probe_ids).logits.float().cpu()
        difference=(before-after).abs().max().item()
        assert difference<1e-3, f'Adapter reload changed logits: {difference}'
        result['reload_max_logit_difference']=difference
    pd.DataFrame(results).to_csv(OUT/'benchmark.csv',index=False)
    del model
    gc.collect()
    if DEVICE=='cuda': torch.cuda.empty_cache()
pd.DataFrame(results)


## Reading the evidence
Use `benchmark.csv` to compare trainable parameters, training time, memory, validation loss and
text overlap. ROUGE and BLEU do not establish ingredient faithfulness or cooking safety: inspect the
saved examples for missing steps, unsupported ingredients and repetitions. General-text loss is a
small forgetting probe, not a broad language benchmark. Parameter savings do not imply identical
runtime savings because frozen layers still perform forward and backward computation.

Run the full dataset before drawing conclusions. Report a failed or worse-performing trial honestly.
The test partition remains untouched. The optional gated Llama extension from the course is outside
this GPT-2 comparison. Add your observed conclusions after execution; no outcomes are invented.

References: [LoRA paper](https://arxiv.org/abs/2106.09685),
[GPT-2 model](https://huggingface.co/openai-community/gpt2), and the course/data links above.
